# Модуль my_utils.py

In [1]:
%%writefile my_utils.py

import functools
import threading
import time
from typing import List, Callable, Any, Dict
import math

# декораторы
def docstring_decorator(description: str):
    """
    Декоратор для добавления и работы с документацией функций.

    Args:
        description (str): Описание функции, которое будет добавлено в docstring

    Returns:
        Callable: Декорированная функция
    """
    def decorator(func: Callable) -> Callable:
        # Сохраняем оригинальный docstring если он есть
        original_doc = func.__doc__ or ""

        # Создаем новый docstring
        func.__doc__ = f"""
{description}

{original_doc}

Дополнительная информация:
- Декорировано с помощью docstring_decorator
- Время создания: {time.ctime()}
"""

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            print(f"Выполняется функция: {func.__name__}")
            print(f"Описание: {description}")
            start_time = time.time()
            result = func(*args, **kwargs)
            end_time = time.time()
            print(f"Функция {func.__name__} выполнена за {end_time - start_time:.4f} секунд")
            return result
        return wrapper
    return decorator

def timer_decorator(func: Callable) -> Callable:
    """
    Декоратор для измерения времени выполнения функции.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"⏱️ Время выполнения {func.__name__}: {end_time - start_time:.4f} секунд")
        return result
    return wrapper

# полезные функции
@docstring_decorator("Вычисляет факториал числа с использованием reduce")
def factorial_reduce(n: int) -> int:
    """
    Вычисляет факториал числа используя функцию reduce.

    Args:
        n (int): Число для вычисления факториала

    Returns:
        int: Факториал числа n
    """
    if n < 0:
        raise ValueError("Факториал определен только для неотрицательных чисел")
    return functools.reduce(lambda x, y: x * y, range(1, n + 1), 1)

@docstring_decorator("Фильтрует простые числа из списка")
def filter_primes(numbers: List[int]) -> List[int]:
    """
    Фильтрует простые числа из переданного списка.

    Args:
        numbers (List[int]): Список чисел для фильтрации

    Returns:
        List[int]: Список простых чисел
    """
    def is_prime(n: int) -> bool:
        if n < 2:
            return False
        for i in range(2, int(math.sqrt(n)) + 1):
            if n % i == 0:
                return False
        return True

    return list(filter(is_prime, numbers))

@docstring_decorator("Применяет функцию к каждому элементу списка в отдельных потоках")
def parallel_map(func: Callable, data: List[Any], num_threads: int = 4) -> List[Any]:
    """
    Параллельно применяет функцию к элементам списка используя потоки.

    Args:
        func (Callable): Функция для применения
        data (List[Any]): Список данных
        num_threads (int): Количество потоков

    Returns:
        List[Any]: Результаты применения функции
    """
    results = [None] * len(data)
    threads = []
    chunk_size = len(data) // num_threads

    def worker(start_idx: int, end_idx: int):
        for i in range(start_idx, end_idx):
            results[i] = func(data[i])

    # Создаем и запускаем потоки
    for i in range(num_threads):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size if i < num_threads - 1 else len(data)
        thread = threading.Thread(target=worker, args=(start_idx, end_idx))
        threads.append(thread)
        thread.start()

    # Ждем завершения всех потоков
    for thread in threads:
        thread.join()

    return results

@docstring_decorator("Вычисляет сумму квадратов чисел используя map и reduce")
def sum_of_squares(numbers: List[int]) -> int:
    """
    Вычисляет сумму квадратов чисел из списка.

    Args:
        numbers (List[int]): Список чисел

    Returns:
        int: Сумма квадратов чисел
    """
    squares = list(map(lambda x: x ** 2, numbers))
    return functools.reduce(lambda x, y: x + y, squares)

@docstring_decorator("Обрабатывает данные в параллельных потоках")
def process_data_parallel(data: List[Any], processing_func: Callable) -> Dict[str, Any]:
    """
    Обрабатывает данные используя многопоточность и возвращает статистику.

    Args:
        data (List[Any]): Данные для обработки
        processing_func (Callable): Функция обработки

    Returns:
        Dict[str, Any]: Статистика обработки
    """
    start_time = time.time()

    # Обрабатываем данные в параллельных потоках
    processed_data = parallel_map(processing_func, data)

    end_time = time.time()

    return {
        'processing_time': end_time - start_time,
        'total_items': len(data),
        'processed_data': processed_data,
        'average_time_per_item': (end_time - start_time) / len(data) if data else 0
    }

# классы
class DataProcessor:
    """
    Класс для обработки данных с переопределенными методами.
    """

    def __init__(self, name: str, data: List[Any] = None):
        self.name = name
        self.data = data or []
        self.processed_data = []
        self.stats = {}

    def __str__(self) -> str:
        """Переопределение строкового представления для пользователя"""
        return f"DataProcessor '{self.name}' с {len(self.data)} элементами данных"

    def __repr__(self) -> str:
        """Переопределение представления для разработчика"""
        return f"DataProcessor(name='{self.name}', data={len(self.data)} items, processed={len(self.processed_data)} items)"

    def __len__(self) -> int:
        """Переопределение длины объекта"""
        return len(self.data)

    def __getitem__(self, index: int) -> Any:
        """Переопределение доступа по индексу"""
        return self.data[index]

    def __contains__(self, item: Any) -> bool:
        """Переопределение проверки вхождения"""
        return item in self.data

    def __call__(self, processing_func: Callable) -> 'DataProcessor':
        """Переопределение вызова объекта как функции"""
        print(f"DataProcessor '{self.name}' вызывается как функция")
        self.processed_data = parallel_map(processing_func, self.data)
        return self

    def add_data(self, item: Any) -> None:
        """Добавляет данные в процессор"""
        self.data.append(item)

    @timer_decorator
    def process(self, func: Callable) -> None:
        """Обрабатывает данные с использованием многопоточности"""
        self.stats = process_data_parallel(self.data, func)
        self.processed_data = self.stats['processed_data']

    def get_statistics(self) -> Dict[str, Any]:
        """Возвращает статистику обработки"""
        return self.stats

    def clear(self) -> None:
        """Очищает все данные"""
        self.data.clear()
        self.processed_data.clear()
        self.stats = {}

# демонстрация функций
def demo_map_reduce_filter():
    """Демонстрирует работу map, reduce, filter"""
    print("=== ДЕМОНСТРАЦИЯ MAP, REDUCE, FILTER ===")

    numbers = list(range(1, 11))
    print(f"Исходные данные: {numbers}")

    # MAP: Удвоение всех чисел
    doubled = list(map(lambda x: x * 2, numbers))
    print(f"MAP (удвоение): {doubled}")

    # FILTER: Только четные числа
    even_numbers = list(filter(lambda x: x % 2 == 0, numbers))
    print(f"FILTER (четные): {even_numbers}")

    # REDUCE: Сумма всех чисел
    total_sum = functools.reduce(lambda x, y: x + y, numbers)
    print(f"REDUCE (сумма): {total_sum}")

    # Комбинация: Сумма квадратов четных чисел
    result = functools.reduce(
        lambda x, y: x + y,
        map(lambda x: x ** 2,
            filter(lambda x: x % 2 == 0, numbers))
    )
    print(f"Комбинация (сумма квадратов четных): {result}")

if __name__ == "__main__":
    demo_map_reduce_filter()

Writing my_utils.py


# Основной ноутбук с использованием модуля

In [2]:
# Импортируем наш модуль
import my_utils as mu
import functools
import time
import random

# Дополнительные импорты для демонстрации
from concurrent.futures import ThreadPoolExecutor
import math

In [3]:
# Демонстрация работы с классом и переопределенными методами

# Создаем экземпляр класса
processor = mu.DataProcessor("МойПроцессор", list(range(1, 21)))

# Демонстрируем переопределенные методы
print("Переопределенные методы:")
print(f"str:  {str(processor)}")
print(f"repr: {repr(processor)}")
print(f"len:  {len(processor)}")
print(f"getitem[5]: {processor[5]}")
print(f"contains(10): {10 in processor}")
print()

# Добавляем данные
processor.add_data(21)
processor.add_data(22)
print(f"После добавления данных: {str(processor)}")

# Используем класс как функцию (__call__)
squared_processor = processor(lambda x: x ** 2)
print(f"После вызова как функции: {repr(squared_processor)}")

Переопределенные методы:
str:  DataProcessor 'МойПроцессор' с 20 элементами данных
repr: DataProcessor(name='МойПроцессор', data=20 items, processed=0 items)
len:  20
getitem[5]: 6
contains(10): True

После добавления данных: DataProcessor 'МойПроцессор' с 22 элементами данных
DataProcessor 'МойПроцессор' вызывается как функция
Выполняется функция: parallel_map
Описание: Применяет функцию к каждому элементу списка в отдельных потоках
Функция parallel_map выполнена за 0.0037 секунд
После вызова как функции: DataProcessor(name='МойПроцессор', data=22 items, processed=22 items)


In [4]:
# Демонстрация декораторов и функций с документацией
print("Факториал через reduce:")
result = mu.factorial_reduce(5)
print(f"Результат: {result}")
print("\nДокументация функции:")
print(mu.factorial_reduce.__doc__)

Факториал через reduce:
Выполняется функция: factorial_reduce
Описание: Вычисляет факториал числа с использованием reduce
Функция factorial_reduce выполнена за 0.0000 секунд
Результат: 120

Документация функции:

Вычисляет факториал числа с использованием reduce


    Вычисляет факториал числа используя функцию reduce.
    
    Args:
        n (int): Число для вычисления факториала
        
    Returns:
        int: Факториал числа n
    

Дополнительная информация:
- Декорировано с помощью docstring_decorator
- Время создания: Sun Nov 30 22:39:34 2025



In [5]:
# Демонтсрация map, reduce, filter

# Запускаем демо из модуля
mu.demo_map_reduce_filter()
print("\nДОПОЛНИТЕЛЬНЫЕ ПРИМЕРЫ:")
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Сложные преобразования с map
cubed_roots = list(map(lambda x: round(x ** (1/3), 2), data))
print(f"Кубические корни: {cubed_roots}")

# Фильтр с условиями
large_primes = list(filter(lambda x: x > 5 and all(x % i != 0 for i in range(2, int(math.sqrt(x)) + 1)), data))
print(f"Простые числа > 5: {large_primes}")

# Reduce для нахождения максимального элемента
max_element = functools.reduce(lambda x, y: x if x > y else y, data)
print(f"Максимальный элемент: {max_element}")

=== ДЕМОНСТРАЦИЯ MAP, REDUCE, FILTER ===
Исходные данные: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
MAP (удвоение): [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
FILTER (четные): [2, 4, 6, 8, 10]
REDUCE (сумма): 55
Комбинация (сумма квадратов четных): 220

ДОПОЛНИТЕЛЬНЫЕ ПРИМЕРЫ:
Кубические корни: [1.0, 1.26, 1.44, 1.59, 1.71, 1.82, 1.91, 2.0, 2.08, 2.15]
Простые числа > 5: [7]
Максимальный элемент: 10


In [6]:
# Демонстрация многопоточности
def complex_calculation(x):
    time.sleep(0.1)  # Имитация сложных вычислений
    return {
        'original': x,
        'square': x ** 2,
        'cube': x ** 3,
        'sqrt': math.sqrt(x),
        'factorial': math.factorial(x) if x < 10 else 0
    }

# Данные для обработки
test_data = list(range(1, 16))

print("Однопоточная обработка:")
start_time = time.time()
single_thread_results = [complex_calculation(x) for x in test_data]
single_thread_time = time.time() - start_time
print(f"Время однопоточной обработки: {single_thread_time:.4f} сек")

print("\nМногопоточная обработка (наш parallel_map):")
multi_thread_results = mu.parallel_map(complex_calculation, test_data, num_threads=4)
print(f"Количество обработанных элементов: {len(multi_thread_results)}")

print("\nМногопоточная обработка (ThreadPoolExecutor):")
start_time = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    thread_pool_results = list(executor.map(complex_calculation, test_data))
thread_pool_time = time.time() - start_time
print(f"Время ThreadPoolExecutor: {thread_pool_time:.4f} сек")

print(f"\nУскорение: {single_thread_time / thread_pool_time:.2f}x")

Однопоточная обработка:
Время однопоточной обработки: 1.5033 сек

Многопоточная обработка (наш parallel_map):
Выполняется функция: parallel_map
Описание: Применяет функцию к каждому элементу списка в отдельных потоках
Функция parallel_map выполнена за 0.6026 секунд
Количество обработанных элементов: 15

Многопоточная обработка (ThreadPoolExecutor):
Время ThreadPoolExecutor: 0.4029 сек

Ускорение: 3.73x


In [8]:
# ДЕмонстрация работы с документацией и декораторами

# Создаем собственную функцию с нашим декоратором
@mu.docstring_decorator("Моя кастомная функция для демонстрации")
def my_custom_function(data):
    """
    Оригинальное описание функции.

    Args:
        data: Входные данные

    Returns:
        Обработанные данные
    """
    time.sleep(0.5)  # Имитация работы
    return [x * 2 + 1 for x in data]

# Используем функцию
test_data = [1, 2, 3, 4, 5]
print("Использование кастомной функции:")
result = my_custom_function(test_data)
print(f"Результат: {result}")

print("\nДокументация функции:")
print(my_custom_function.__doc__)

Использование кастомной функции:
Выполняется функция: my_custom_function
Описание: Моя кастомная функция для демонстрации
Функция my_custom_function выполнена за 0.5002 секунд
Результат: [3, 5, 7, 9, 11]

Документация функции:

Моя кастомная функция для демонстрации


    Оригинальное описание функции.
    
    Args:
        data: Входные данные
        
    Returns:
        Обработанные данные
    

Дополнительная информация:
- Декорировано с помощью docstring_decorator
- Время создания: Sun Nov 30 22:49:14 2025



In [10]:
# Бенчмарк производительности

def benchmark_processing():
    large_dataset = list(range(1, 1001))

    # Функция для обработки
    def process_item(x):
        time.sleep(0.001)  # Небольшая задержка
        return x ** 2 + math.sin(x) + math.log(x + 1)

    # 1. Обычный map (однопоточный)
    start_time = time.time()
    result1 = list(map(process_item, large_dataset))
    time1 = time.time() - start_time

    # 2. parallel_map
    start_time = time.time()
    result2 = mu.parallel_map(process_item, large_dataset, num_threads=8)
    time2 = time.time() - start_time

    # 3. ThreadPoolExecutor
    start_time = time.time()
    with ThreadPoolExecutor(max_workers=8) as executor:
        result3 = list(executor.map(process_item, large_dataset))
    time3 = time.time() - start_time

    print("Результаты бенчмарка (1000 элементов):")
    print(f"Однопоточный map:    {time1:.4f} сек")
    print(f"Наш parallel_map:     {time2:.4f} сек")
    print(f"ThreadPoolExecutor:   {time3:.4f} сек")
    print(f"Ускорение parallel_map: {time1/time2:.2f}x")
    print(f"Ускорение ThreadPool:  {time1/time3:.2f}x")

    # Проверяем корректность результатов
    assert result1 == result2 == result3, "Результаты должны быть идентичными"
    print("Все методы дали одинаковые результаты")

benchmark_processing()

Выполняется функция: parallel_map
Описание: Применяет функцию к каждому элементу списка в отдельных потоках
Функция parallel_map выполнена за 0.1416 секунд
Результаты бенчмарка (1000 элементов):
Однопоточный map:    1.1263 сек
Наш parallel_map:     0.1419 сек
ThreadPoolExecutor:   0.1512 сек
Ускорение parallel_map: 7.94x
Ускорение ThreadPool:  7.45x
Все методы дали одинаковые результаты
